# BIF201

## Reference genome, alignment, and variant calling

**BIF201 Working Version: v2.2**
**Instructor: Önder Akyün**

This notebook was prepared so you can re-experience, in your own working environment, the analysis pipeline we built together during the live lesson. Our aim here goes beyond simply repeating the commands — it's to reconstruct, within your own workflow, the origin of every file, the meaning of every output, and the connections between each step.

**First exercise**

We will work with real FLW-1 reads that have already been through quality control, trimming, filtering, denoising, and re-quality-control. We will align the Illumina and Oxford Nanopore data separately to the reference assembly of the same isolate, and then read together the structure and quality of the resulting BAM outputs.

**Second exercise**

We will then move on to training data that preserves the same read structure but carries 40 controlled SNVs. We will complete the alignments performed on this data together with variant calling, truth-VCF comparison, and performance evaluation in the next live lesson.

In these two exercises we are pursuing two separate questions:

1. What do we see when we align the same isolate's natural reads to its own reference assembly?
2. To what extent does the caller recover the signal when we plant SNVs we already know into the reads?

**Accompanying document:** `BIF201_Calisma_Surumu_v2.1_Uygulama_Kilavuzu.docx`

## Starting point

We are not starting this work from raw data. We completed raw-data quality control and denoising in previous exercises. Instead of redoing those steps today, we will carry our already-processed FASTQ files to the next step of the analysis chain.

For this work, it is enough to have the following three processed FASTQ files ready:

```text
filtered_SRR13680736_1.fastq
filtered_SRR13680736_2.fastq
filtered_ONT.fastq
```

If the files are stored as `.fastq.gz`, `.fq`, or `.fq.gz`, the notebook will also recognize these extensions.

Our first new task is to bring the FLW-1 reference assembly into the working environment. We will build the reads' positions on the genome, the BAM records, and the VCF coordinates we will produce later, all on this shared reference space.



```
# This is formatted as code
```

## Our work plan

### Stages we will cover

1. Obtaining the FLW-1 reference assembly from NCBI
2. Verifying the contig structure and total length of the reference
3. Linking the previously prepared FASTQ files to their real roles
4. Building the BWA and Minimap2 indexes
5. Aligning the Illumina reads
6. Aligning the Nanopore reads
7. Producing BAM quality-control reports


### Stages we will cover

0. Starting variant calling on the natural Illumina and Nanopore data
1. Reading the natural data calls together
2. Truth VCF comparison
3. Calculating TP, FN, and FP values
4. Interpreting sensitivity and precision together
5. Inspecting selected positions in IGV
6. Normalization and initial filters for the controlled (variants I manipulated and planted) variant calls



## Analytical expectation

The FLW-1 reference assembly was produced from the same isolate's Illumina and Nanopore reads. For this reason, we expect the number of high-confidence SNVs in the natural data to be close to zero.

We are not establishing an absolute assumption of zero here. Low-frequency heterogeneity, assembly consensus, alignment ambiguity, base errors, and caller behavior can all produce VCF records.

Keep this distinction in mind throughout the lesson:

```text
VCF record = a candidate produced by the caller
biological variant = an interpretation supported by additional evidence
```

# EXERCISES 1-59: REFERENCE, ALIGNMENT, AND BAM QUALITY CONTROL

The notebook's first 59 exercises were prepared so you can rebuild, in your own working environment, the reference acquisition, alignment, and BAM quality-control process we carried out together during the live lesson.

This work spans exercises 1 through 59. The comparison and variant-calling sections beginning with exercise 60 belong to the next stage of work.

Inside the accompanying **BIF201 Working Version v2.1 Exercise Guide**, separate paths are provided for data stored on your computer, data stored on Google Drive, renamed folders, and renamed FASTQ files.


## Let's clarify our data situation before we start

| Data situation | Path to follow |
|---|---|
| FASTQ files are on Google Drive | `DATA_LOCATION = "drive"` |
| FASTQ files are on your computer, to be uploaded to Drive first | After uploading, `DATA_LOCATION = "drive"` |
| FASTQ files will be uploaded directly to the Colab session | `DATA_LOCATION = "computer"` |
| File or folder names have changed | You can use the real paths in the candidate file table |
| File names kept their original form | The notebook can link the files automatically |

For files at gigabyte scale, I recommend the Google Drive path. Colab's `/content` space is temporary and gets deleted when the session ends.

## The shared framework we'll follow throughout the work

1. We don't need to redo the raw-data quality control and denoising steps.
2. As input, we will use the already-processed FASTQ files produced earlier.
3. Rather than accepting the R1, R2, and Nanopore roles based on file name alone, we will verify them with checks inside the notebook.
4. When an error occurs, recording the cell number together with the full error output makes it easier for us to jointly find the source of the problem.
5. Exporting the BAM, BAI, and quality-control reports to a permanent location before the session ends protects the work.


# PART I: Working environment

First we organize the workspace. In this part we'll set up the Colab connection, the data search roots, and the output folders. We are not yet processing any biological data.

## 1. Google Drive connection

If the FASTQ files are on Google Drive, we will first connect Colab to Drive. Once authorization is complete, the Drive content becomes visible under `/content/drive/MyDrive`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Note for those keeping their data on their computer

The Google Drive connection can still be set up, so that the analysis outputs can be moved to a permanent location at the end of the lesson.

If you prefer to upload FASTQ files directly from your computer to Colab, you can use the **Files** section in the left sidebar, and upload files into the `/content/BIF201_local_uploads` folder. If the files are large, uploading through the browser can take a long time. In that case, uploading to Google Drive first is the safer option.

## 2. Importing the Python file-system tool

We could also write file paths as plain text, but in long and variable folder structures the risk of error grows. So we'll use `Path`, to build file paths in a safer and more readable way.

In [ ]:
from pathlib import Path
import shutil

## 3. Defining the data source and search roots

In this cell you will explicitly state where your data is located. `DATA_LOCATION` accepts only two values:

- `"drive"`: we'll search for your FASTQ files in Google Drive or a shared Drive.
- `"computer"`: we'll search under `/content` for the FASTQ files you uploaded from your computer to the Colab session.

It's not a problem if you've renamed a folder or file. We'll resolve the real R1, R2, and Nanopore roles together in exercises 24-27.

In [ ]:
DATA_LOCATION = "drive"  # "drive" or "computer"
LOCAL_UPLOAD_DIR = Path("/content/BIF201_local_uploads"); LOCAL_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ROOTS = [p for p in [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")] if p.exists()]
SEARCH_ROOTS = DRIVE_ROOTS if DATA_LOCATION == "drive" else [LOCAL_UPLOAD_DIR, Path("/content")]
INPUT_DIR = SEARCH_ROOTS[0]

### Adding a specific folder to the search

If the files are in an unusual Drive subfolder, or in a different location in Colab, you can add the full folder path to the list below.

Example:

```python
EXTRA_SEARCH_ROOTS = ["/content/drive/MyDrive/thesis_data/final_fastq"]
```

Using the exact path shown in the Colab file panel, rather than guessing the folder name, prevents the wrong file from being selected.

In [ ]:
EXTRA_SEARCH_ROOTS = []
custom_roots = [Path(p) for p in EXTRA_SEARCH_ROOTS if Path(p).exists()]
if custom_roots: SEARCH_ROOTS = custom_roots
print("Search roots:", *SEARCH_ROOTS, sep="\n- ")

### Direct upload-from-computer check

When `DATA_LOCATION = "computer"` is selected, this cell lists the files that appear in the Colab session after upload. Seeing the FASTQ files in the list is the basic check we need before moving on to exercise 24.

In [ ]:
if DATA_LOCATION == "computer":
    print("Local upload folder:", LOCAL_UPLOAD_DIR)
    print(*[p for p in Path("/content").rglob("*") if p.is_file()][:50], sep="\n")

## 4. Defining the analysis root

We will collect all new outputs under a single analysis root. This way, we won't confuse which run a given reference, BAM, report, or variant file belongs to.

When the Colab session ends, the `/content` space is deleted. That's why we will copy the selected outputs to Drive at the end of the lesson.

In [ ]:
ROOT = Path("/content/BIF201_referans_hizalama")

## 5. Defining subfolders

We will keep the reference, alignment, quality-control, variant, comparison, and log files in separate folders. This structure will make it easier for us to later reconstruct which stage a given file was produced in.

In [ ]:
REF_DIR, BAM_DIR, QC_DIR = ROOT/"reference", ROOT/"alignments", ROOT/"mapping_qc"
VAR_DIR, COMP_DIR, LOG_DIR = ROOT/"variants", ROOT/"comparison", ROOT/"logs"

## 6. Creating the working folders

Now we physically create the folder structure we defined. We'll also open any missing parent folders along the way; we won't rename folders that already exist.

In [ ]:
for folder in [ROOT, REF_DIR, BAM_DIR, QC_DIR, VAR_DIR, COMP_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

## 7. Checking disk space

Full-size FASTQ and BAM files can use several gigabytes of space. Seeing the free space on Colab's temporary disk before alignment helps us assess the risk of the process stalling halfway.

In [ ]:
!df -h /content

# PART II: Software environment

In this exercise we will use BWA, Minimap2, SAMtools, and BCFtools. Each tool takes on its own part of the process:

- BWA: aligning Illumina short reads
- Minimap2: aligning Nanopore long reads
- SAMtools: BAM sorting, indexing, and quality control
- BCFtools: pileup, variant calling, normalization, and comparison

## 8. Installing command-line tools

In this cell we will install BWA, Minimap2, SAMtools, BCFtools, and helper commands into the Colab runtime. This installation is valid for the current open session; if you open a new runtime, you'll need to rerun this cell.

In [ ]:
!apt-get update -qq
!apt-get install -y bwa minimap2 samtools bcftools curl zip

## 9. Installing Python analysis packages

We will read result tables in Python with `pandas`, and FASTA and VCF files with `pysam`. We're installing these packages into the current Colab session now.

In [ ]:
!pip -q install pandas pysam

## 10. Recording the BWA version

We record the software versions we're using right from the start; reproducibility begins with this. We'll also write the first lines of the BWA output to the log file.

In [ ]:
!bwa 2>&1 | head -n 5 | tee "{LOG_DIR/'bwa.version.txt'}"

## 11. Recording the Minimap2 version

We record this information now so we can trace it later.

We will store the Minimap2 version we use for long-read alignment in a separate file.

In [ ]:
!minimap2 --version | tee "{LOG_DIR/'minimap2.version.txt'}"

## 12. Recording the SAMtools version

We record this information now so we can trace it later.

Alongside the SAMtools version, we will also report the HTSlib version.

In [ ]:
!samtools --version | head -n 3 | tee "{LOG_DIR/'samtools.version.txt'}"

## 13. Recording the BCFtools version

We record this information now so we can trace it later.

We will produce the variant calling and set-comparison results with the BCFtools version we record here.

In [ ]:
!bcftools --version | head -n 3 | tee "{LOG_DIR/'bcftools.version.txt'}"

# PART III: Obtaining the FLW-1 reference assembly

Now we set up the shared coordinate space onto which we will place the reads. The FLW-1 reference assembly contains four contigs. We will read the alignment records and the future variant coordinates on these four sequences.

Contig accession numbers we will use:

```text
JAFEJB010000001.1
JAFEJB010000002.1
JAFEJB010000003.1
JAFEJB010000004.1
```

## 14. Defining the reference FASTA path

First, in this step, we explicitly define the file paths we will use.

We will keep the four contigs we get from NCBI in a single FASTA file.

In [ ]:
REFERENCE = REF_DIR / "G_cerinus_FLW1_original.fasta"

## 15. Defining the contig accession numbers

Now we explicitly define the contig accession numbers we will use.

We will write the four accession numbers separated by commas, in the format required by the NCBI EFetch request.

In [ ]:
CONTIG_IDS = "JAFEJB010000001.1,JAFEJB010000002.1,JAFEJB010000003.1,JAFEJB010000004.1"

## 16. Setting up the NCBI EFetch address

In this cell we set up the NCBI query address.

When we call this address, the NCBI Nucleotide database will return the sequences in FASTA format.

In [ ]:
REFERENCE_URL = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id={CONTIG_IDS}&rettype=fasta&retmode=text"

## 17. Downloading the reference assembly

Now we download the data into the working directory.

`curl` will retry the request if the connection drops. We'll write the downloaded content directly to the reference FASTA path we defined.

In [ ]:
!curl -L --retry 3 --fail -o "{REFERENCE}" "{REFERENCE_URL}"

## 18. Basic check of the FASTA file

Before continuing processing, we check that this step has actually completed.

We'll check that the file was created and is larger than zero bytes. If the download failed, we'll stop the analysis here; we won't proceed with a corrupted reference.

In [ ]:
assert REFERENCE.exists() and REFERENCE.stat().st_size > 0
print(f"Reference size: {REFERENCE.stat().st_size/1_000_000:.2f} MB")

## 19. Inspecting the FASTA headers

We'll read this output together and evaluate whether the structure we expect has formed.

The four contig headers appear on screen. We expect the accession numbers to appear with the `.1` version suffix.

In [ ]:
!grep '^>' "{REFERENCE}"

## 20. Building the FASTA index

Now we build the corresponding index; the subsequent tools will build region access on this structure.

We will produce the `.fai` file carrying the contig names and lengths, using `samtools faidx`.

In [ ]:
!samtools faidx "{REFERENCE}"

## 21. Displaying the contig lengths

This cell reveals the first two columns of the FAI file. In the output, the contig name and length appear side by side.

In [ ]:
!cut -f1,2 "{REFERENCE}.fai"

## 22. Verifying the assembly structure

In this step we verify that the conditions we expect are actually met.

We expect to see four contigs and a total of 3,328,849 bases in the FLW-1 reference. We will verify both conditions together in the same cell.

In [ ]:
fai = Path(str(REFERENCE) + ".fai")
rows = [line.rstrip().split("\t") for line in fai.open()]
total_length = sum(int(row[1]) for row in rows)
assert len(rows) == 4 and total_length == 3_328_849
print(f"Contig count: {len(rows)} | Total length: {total_length}")

## 23. Recording the reference digest

We record this information now so we can trace it later.

We'll store the SHA-256 digest of the FASTA copy we're using so we can definitively identify it later.

In [ ]:
!sha256sum "{REFERENCE}" | tee "{LOG_DIR/'reference.sha256.txt'}"

# PART IV: FASTQ files carried over from the previous exercise

In this part we are not redoing quality control. We will find the three processed FASTQ files you produced in previous lessons, link them to their real roles, and verify they are ready for alignment.

## 24. Helper function that finds FASTQ candidates

Here we build a small helper function that finds FASTQ candidates without blindly trusting the file name.

The notebook scans the following extensions together:

- `.fastq`
- `.fastq.gz`
- `.fq`
- `.fq.gz`

We'll run the scan across the root folders we defined in exercise 3. Even if you've renamed a file, the FASTQ candidate should appear in the table.

In [ ]:
FASTQ_SUFFIXES = (".fastq", ".fastq.gz", ".fq", ".fq.gz")
def discover_fastqs(roots):
    return sorted({p for root in roots for suffix in FASTQ_SUFFIXES
                   for p in root.rglob(f"*{suffix}") if p.is_file() and p.stat().st_size > 0})

## 25. Scanning the FASTQ candidates

Now we scan the files we can access; at this stage we don't yet assign a biological role.

In this cell we don't yet assign an R1, R2, or Nanopore role. We are only finding the FASTQ files we can access.

If there are many FASTQ files in Google Drive, the scan can take a few minutes.

In [ ]:
FASTQ_CANDIDATES = discover_fastqs(SEARCH_ROOTS)
assert FASTQ_CANDIDATES, "No FASTQ found in the search roots. Check the data source and upload location."
assert len(FASTQ_CANDIDATES) <= 200, "More than 200 FASTQ files found. Narrow the search to the project folder with EXTRA_SEARCH_ROOTS."
print("FASTQ candidates found:", len(FASTQ_CANDIDATES))

### Preparing the FASTQ candidate table

This table contains each candidate file's full path, size, first read ID, and the median read length across the first 100 records.

If a file or folder name has been changed, the decision can be made using this table. The file name alone is not sufficient evidence; reading the path, size, and read structure together is safer.

In [ ]:
import itertools, pandas as pd, pysam
def fastq_preview(path, n=100):
    with pysam.FastxFile(str(path)) as handle: records = list(itertools.islice(handle, n))
    lengths = [len(r.sequence) for r in records]
    return {"PATH": str(path), "SIZE_GB": round(path.stat().st_size/1e9, 3), "FIRST_ID": records[0].name if records else "", "MEDIAN_LENGTH": pd.Series(lengths).median()}

In [ ]:
FASTQ_TABLE = pd.DataFrame([fastq_preview(p) for p in FASTQ_CANDIDATES])
FASTQ_TABLE.index.name = "CANDIDATE_NO"
display(FASTQ_TABLE)

### How will we distinguish the file roles?

**Illumina R1 and R2**

- Two files belonging to the same paired-end library.
- The record counts are expected to be equal.
- The first 1,000 read IDs must match.
- Read lengths are usually very close to each other.

**Nanopore**

- A single long-read file.
- Read lengths are more variable than Illumina reads.
- Having `ONT` or `Nanopore` in the file name is only a hint. The exact role needs to be verified through the content.

If file names have been changed, the `PATH` values in the candidate table can be used in the manual path fields below.

## 26. Manual path fields for renamed files

First, in this step, we explicitly define the file paths we will use.

If your file names kept their original form, leave the three fields blank; the notebook will search for the standard names itself.

If you renamed the files, paste the full path from the candidate table exactly into the relevant field. Don't shorten the folder path or delete the file extension.

In [ ]:
MANUAL_R1 = ""
MANUAL_R2 = ""
MANUAL_ONT = ""

### Automatic and manual path resolution

If you filled in the manual field, the notebook uses the path you gave directly. If the field is empty, it searches for the original file name among the four FASTQ extensions.

If more than one file with the same name is found, the notebook will ask you to make a manual selection. This way we avoid mixing up an old backup file with the current working file.

In [ ]:
def resolve_fastq(manual, stem):
    if manual.strip(): return Path(manual.strip())
    names = {f"{stem}{suffix}".lower() for suffix in FASTQ_SUFFIXES}
    matches = [p for p in FASTQ_CANDIDATES if p.name.lower() in names]
    return matches[0] if len(matches) == 1 else None

In [ ]:
ILLUMINA_R1 = resolve_fastq(MANUAL_R1, "filtered_SRR13680736_1")
ILLUMINA_R2 = resolve_fastq(MANUAL_R2, "filtered_SRR13680736_2")
NANOPORE_FASTQ = resolve_fastq(MANUAL_ONT, "filtered_ONT")

## 27. Linking the three FASTQ roles

In this step we link the R1, R2, and Nanopore roles to their real file paths.

This cell verifies that all three roles are linked to a single file. If an error occurs, enter the full paths in the manual fields from exercise 26.

We don't allow the same file to be linked to two different roles.

In [ ]:
resolved = {"Illumina R1": ILLUMINA_R1, "Illumina R2": ILLUMINA_R2, "Nanopore": NANOPORE_FASTQ}
unresolved = [role for role, path in resolved.items() if path is None]
assert not unresolved, "Manual path required: " + ", ".join(unresolved)
assert len({str(p.resolve()) for p in resolved.values()}) == 3, "The same file is assigned to more than one role."
print(*[f"{role} -> {path}" for role, path in resolved.items()], sep="\n")

## 27A. File existence and readability check

In this step we verify that the conditions we expect are actually met.

After completing the role assignment, we'll check that the files actually exist and are not empty.

In [ ]:
FASTQS = [ILLUMINA_R1, ILLUMINA_R2, NANOPORE_FASTQ]
missing = [p for p in FASTQS if not p.exists() or p.stat().st_size == 0]
assert not missing, "Missing FASTQ:\n" + "\n".join(map(str, missing))
print("All three FASTQ files were found and are readable.")

## 27B. Recording the selected inputs

We record this information now so we can trace it later.

We will write the real paths, file sizes, and data roles we used to a TSV file. This record will show which inputs we used for the analysis, even if you've renamed a folder or file.

In [ ]:
input_manifest = pd.DataFrame({"ROLE": list(resolved), "PATH": [str(p) for p in resolved.values()],
                               "SIZE_GB": [round(p.stat().st_size/1e9, 3) for p in resolved.values()]})
display(input_manifest)
input_manifest.to_csv(LOG_DIR/"selected_input_manifest.tsv", sep="\t", index=False)

## 27C. Optional discovery of previous quality-control reports

Before continuing processing, we check that this step has actually been completed.

FastQC, MultiQC, NanoPlot, or similar reports produced in previous lessons are not among the computational inputs of this pipeline. Finding the reports is useful for tracking the data's history and choosing the correctly processed file.

If you'd like to search for the reports, set `DISCOVER_REPORTS = True`. In large folders, this scan can take time.

In [ ]:
DISCOVER_REPORTS = True
REPORT_KEYS = ("fastqc", "multiqc", "nanoplot", "report", "qc")
report_roots = {p.parent for p in FASTQS}
REPORTS = sorted({p for root in report_roots for p in root.rglob("*") if p.is_file() and any(k in p.name.lower() for k in REPORT_KEYS)})[:200] if DISCOVER_REPORTS else []
print("Reports found:", len(REPORTS)); print(*REPORTS, sep="\n")

## 28. Displaying file sizes

In this cell we make the output visible and read it together.

File sizes will quickly help you see whether you've selected the correct inputs.

In [ ]:
for path in FASTQS:
    print(f"{path.name}: {path.stat().st_size/1_000_000_000:.2f} GB")

## 29. Illumina R1 record count

In this cell we compute the relevant record count.

We will directly use the fact that a FASTQ record consists of four lines in this count. With `zcat -f` we can read both compressed and plain-text FASTQ files.

In [ ]:
import subprocess

cmd = f'zcat -f "{ILLUMINA_R1}" | wc -l'
line_count = int(subprocess.check_output(cmd, shell=True, text=True))

assert line_count % 4 == 0
print("Illumina R1:", line_count // 4, "records")

## 30. Illumina R2 record count

In this cell we compute the relevant record count.

We expect the R1 and R2 record counts to be equal.

In [ ]:
cmd = f'zcat -f "{ILLUMINA_R2}" | wc -l'
line_count = int(subprocess.check_output(cmd, shell=True, text=True))

assert line_count % 4 == 0
print("Illumina R2:", line_count // 4, "records")

## 31. Nanopore record count

In this cell we compute the relevant record count.

This output will give us the number of independent Nanopore reads.

In [ ]:
cmd = f'zcat -f "{NANOPORE_FASTQ}" | wc -l'
line_count = int(subprocess.check_output(cmd, shell=True, text=True))

assert line_count % 4 == 0
print("Nanopore:", line_count // 4, "records")

## 32. ID check of the first 1,000 Illumina pairs

Before continuing processing, we check that this step has actually completed.

We'll compare the first fields of the R1 and R2 headers. If there's a mismatch within the first 1,000 pairs, the cell will stop with an error.

In [ ]:
import itertools, pysam
r1 = [r.name.removesuffix("/1") for r in itertools.islice(pysam.FastxFile(str(ILLUMINA_R1)), 1000)]
r2 = [r.name.removesuffix("/2") for r in itertools.islice(pysam.FastxFile(str(ILLUMINA_R2)), 1000)]
assert len(r1) == len(r2) == 1000 and r1 == r2; print("The first 1,000 Illumina pairs matched.")

### 32A. How should we read the ID check?

The first 1,000 pairs matching strongly supports the idea that the R1 and R2 files belong to the same paired-end library. This check alone does not prove that the entire file is complete. We'll also read the record count, file size, and alignment statistics together.

When a matching error occurs, two possibilities stand out:

1. The R1 and R2 roles may have been linked to the wrong files.
2. One of the files may come from a different filtering run or a different sample.

# PART V: Reference indexes

We are not changing the reference FASTA sequence. BWA and Minimap2 will convert the same sequence into two different index structures, each suited to its own search method.

## 33. Building the BWA indexes

Now we build the corresponding index; the subsequent tools will build region access on this structure.

We will produce, alongside the reference FASTA, the index files BWA will use for short-read search.

In [ ]:
!bwa index "{REFERENCE}"

## 34. Building the Minimap2 index

Now we build the corresponding index; the subsequent tools will build region access on this structure.

We will produce the `.mmi` file Minimap2 will use for long-read search.

In [ ]:
MINIMAP2_INDEX = Path(str(REFERENCE) + ".mmi")
!minimap2 -d "{MINIMAP2_INDEX}" "{REFERENCE}"

## 35. Inspecting the reference folder

This cell reveals the FASTA, FAI, BWA indexes, and MMI file within the same folder. For alignment to proceed, this group of files needs to be complete.

In [ ]:
!ls -lh "{REF_DIR}"

# PART VI: Aligning the Illumina reads

Now we will align the Illumina paired-end 2nd-generation sequencing (NGS) reads to the reference with BWA-MEM. The orientation of the mates and their distance on the reference are important parts of the alignment decision.

We will add the following information to the BAM header:

- technical read group
- biological sample name
- platform
- library name

## Before moving on to Illumina alignment

Before starting the alignment cell, it's important that four conditions are met together:

- the reference FASTA and FAI files are ready,
- the BWA indexes are ready,
- the R1 and R2 record counts are equal,
- the IDs of the first 1,000 pairs match.

If one of the conditions isn't met, re-evaluating that step first protects the reliability of the alignment result.

## 36. Defining the Illumina BAM paths

First, in this step, we explicitly define the file paths we will use.

We will keep the unsorted BAM, the sorted BAM, and the index in the same folder.

In [ ]:
ILL_UNSORTED = BAM_DIR/"natural_illumina.unsorted.bam"
ILL_BAM = BAM_DIR/"natural_illumina.sorted.bam"

## 37. Aligning the Illumina reads with BWA-MEM

In this cell we align the Illumina reads to the reference. Once the command completes, we'll read the BAM file together with the alignment log.

BWA-MEM first produces a SAM stream. `samtools view` writes this stream to disk in BAM format. The alignment log is kept in a separate file.

In [ ]:
!bwa mem -t 2 -R '@RG\tID:FLW1_ILL\tSM:FLW1_NATURAL\tPL:ILLUMINA\tLB:BIF201_ILL' "{REFERENCE}" "{ILLUMINA_R1}" "{ILLUMINA_R2}" 2> "{LOG_DIR/'natural_illumina.bwa.log'}" | samtools view -b -o "{ILL_UNSORTED}" -

## 38. Checking the unsorted Illumina BAM

Before continuing processing, we check that this step has actually completed.

With `samtools quickcheck` we'll quickly check the BAM header and the end of the file. If the command completes silently, this check has passed.

In [ ]:
!samtools quickcheck -v "{ILL_UNSORTED}"

## 39. Sorting the Illumina BAM by coordinate

Now we sort the BAM records by reference coordinate.

We will put the records in contig and genome coordinate order. Variant calling and IGV use this ordering.

In [ ]:
!samtools sort -@ 2 -o "{ILL_BAM}" "{ILL_UNSORTED}"

## 40. Checking the sorted Illumina BAM

In this step we verify that the conditions we expect are actually met.

Once sorting is finished, we'll check the new BAM once more.

In [ ]:
!samtools quickcheck -v "{ILL_BAM}"

## 41. Building the Illumina BAM index

Now we build the corresponding index; the subsequent tools will build region access on this structure.

The BAI index will let us directly access specific genome ranges.

In [ ]:
!samtools index -@ 2 "{ILL_BAM}"

## 42. Inspecting the Illumina read group record

This cell reveals the `@RG` line in the BAM header. We expect the sample name, platform, and library information to match the values we gave in the command.

In [ ]:
!samtools view -H "{ILL_BAM}" | grep '^@RG'

## 43. Illumina flagstat report

With this report we extract a basic summary of the alignment. We'll evaluate the total record count, aligned reads, properly paired pairs (`properly paired`), and singleton count together.

In [ ]:
!samtools flagstat -@ 2 "{ILL_BAM}" | tee "{QC_DIR/'natural_illumina.flagstat.txt'}"

## 44. Illumina coverage report

Now we'll read the coverage and depth values per contig.

We will compute the percentage of bases covered, average depth, average base quality, and MAPQ value for each contig.

In [ ]:
!samtools coverage "{ILL_BAM}" | tee "{QC_DIR/'natural_illumina.coverage.tsv'}"

## 45. Illumina idxstats report

In this cell we extract the number of aligned records per contig.

We'll summarize the number of records aligned to each contig from the BAM index.

In [ ]:
!samtools idxstats "{ILL_BAM}" | tee "{QC_DIR/'natural_illumina.idxstats.tsv'}"

## 46. Illumina detailed statistics report

We write more detailed alignment statistics to a separate report.

We will record read length, insert size, and alignment summaries in the form of a detailed report.

In [ ]:
!samtools stats -@ 2 "{ILL_BAM}" > "{QC_DIR/'natural_illumina.stats.txt'}"

## 47. Removing the intermediate Illumina BAM file

After verifying the sorted BAM and its index, we remove the intermediate file we no longer need.

Once we've verified the sorted BAM and BAI file, we'll remove the unsorted intermediate BAM to save disk space.

In [ ]:
!rm -f "{ILL_UNSORTED}"

# PART VII: Aligning the Nanopore reads

In this part we'll align the Nanopore long reads using Minimap2's `map-ont` preset. A single long read can cover a wide genome range within one record. Some reads produce split alignments; these records appear as supplementary within the BAM.

## Before moving on to Nanopore alignment

The Minimap2 `.mmi` index, the Nanopore FASTQ file, and sufficient disk space need to be ready.

The Nanopore BAM carries a different record structure from the Illumina BAM. Empty paired-end fields are consistent with the singleton-record structure of long reads.

## 48. Defining the Nanopore BAM paths

First, in this step, we explicitly define the file paths we will use.

We will name the Nanopore outputs so they're clearly distinguished from the Illumina files.

In [ ]:
ONT_UNSORTED = BAM_DIR/"natural_nanopore.unsorted.bam"
ONT_BAM = BAM_DIR/"natural_nanopore.sorted.bam"

## 49. Aligning the Nanopore reads with Minimap2

In this cell we align the Nanopore reads to the reference. Once the command completes, we'll read the BAM file together with the alignment log.

`map-ont` uses settings suited to the Nanopore error profile. Secondary alignments are excluded; primary and required supplementary records are kept.

In [ ]:
!minimap2 -ax map-ont --secondary=no -t 2 -R '@RG\tID:FLW1_ONT\tSM:FLW1_NATURAL\tPL:ONT\tLB:BIF201_ONT' "{MINIMAP2_INDEX}" "{NANOPORE_FASTQ}" 2> "{LOG_DIR/'natural_nanopore.minimap2.log'}" | samtools view -b -o "{ONT_UNSORTED}" -

## 50. Checking the unsorted Nanopore BAM

In this step we verify that the conditions we expect are actually met.

We'll quickly check the BAM header and the end of the file.

In [ ]:
!samtools quickcheck -v "{ONT_UNSORTED}"

## 51. Sorting the Nanopore BAM by coordinate

Now we sort the BAM records by reference coordinate.

We'll rebuild the record order according to reference coordinate.

In [ ]:
!samtools sort -@ 2 -o "{ONT_BAM}" "{ONT_UNSORTED}"

## 52. Checking the sorted Nanopore BAM

In this step we verify that the conditions we expect are actually met.

We'll verify the sorted BAM before indexing it.

In [ ]:
!samtools quickcheck -v "{ONT_BAM}"

## 53. Building the Nanopore BAM index

Now we build the corresponding index; the subsequent tools will build region access on this structure.

We'll use the BAI file for regional access and IGV inspection.

In [ ]:
!samtools index -@ 2 "{ONT_BAM}"

## 54. Inspecting the Nanopore read group record

This cell reveals the read group record in the BAM header. We expect `PL:ONT` and the sample name to appear in the output.

In [ ]:
!samtools view -H "{ONT_BAM}" | grep '^@RG'

## 55. Nanopore flagstat report

This report gives a basic summary of the alignment.

Primary and supplementary records appear in the same report. Zero paired-end fields are consistent with Nanopore's singleton-read structure.

In [ ]:
!samtools flagstat -@ 2 "{ONT_BAM}" | tee "{QC_DIR/'natural_nanopore.flagstat.txt'}"

## 56. Nanopore coverage report

Now we'll read the coverage and depth values per contig.

We'll compute per-contig coverage and average depth.

In [ ]:
!samtools coverage "{ONT_BAM}" | tee "{QC_DIR/'natural_nanopore.coverage.tsv'}"

## 57. Nanopore idxstats report

In this cell we extract the number of aligned records per contig.

We'll summarize the number of aligned records on each contig.

In [ ]:
!samtools idxstats "{ONT_BAM}" | tee "{QC_DIR/'natural_nanopore.idxstats.tsv'}"

## 58. Nanopore detailed statistics report

We write more detailed alignment statistics to a separate report.

We'll write long-read length distributions and alignment summaries to file.

In [ ]:
!samtools stats -@ 2 "{ONT_BAM}" > "{QC_DIR/'natural_nanopore.stats.txt'}"

## 59. Removing the intermediate Nanopore BAM file

After verifying the sorted BAM and its index, we remove the intermediate file we no longer need.

Once the sorted BAM and index are ready, we'll remove the intermediate file.

In [ ]:
!rm -f "{ONT_UNSORTED}"

# EXERCISES 1-59 COMPLETION POINT

We complete exercises 1-59 here.

At this point the following core outputs are expected to have been produced:

```text
natural_illumina.sorted.bam
natural_illumina.sorted.bam.bai
natural_nanopore.sorted.bam
natural_nanopore.sorted.bam.bai
natural_illumina.flagstat.txt
natural_illumina.coverage.tsv
natural_illumina.idxstats.tsv
natural_illumina.stats.txt
natural_nanopore.flagstat.txt
natural_nanopore.coverage.tsv
natural_nanopore.idxstats.tsv
natural_nanopore.stats.txt
selected_input_manifest.tsv
```

Checks 59A-59D complete the integrity check and export to permanent storage for these outputs. This work ends at 59D; exercise 60 is reserved for the next stage.


## 59A. Verifying the output contract

In this step we verify that the conditions we expect are actually met.

We'll check the existence of the BAM, BAI, and quality-control reports in a single table.

In [ ]:
STEP59_OUTPUTS = [ILL_BAM, Path(str(ILL_BAM)+".bai"), ONT_BAM, Path(str(ONT_BAM)+".bai")]
STEP59_OUTPUTS += [QC_DIR/f"natural_{p}.{r}" for p in ("illumina","nanopore") for r in ("flagstat.txt","coverage.tsv","idxstats.tsv","stats.txt")]
step59_audit = pd.DataFrame({"PATH": [str(p) for p in STEP59_OUTPUTS], "EXISTS": [p.exists() for p in STEP59_OUTPUTS],
                             "SIZE_MB": [round(p.stat().st_size/1e6, 2) if p.exists() else 0 for p in STEP59_OUTPUTS]})
display(step59_audit)

## 59B. BAM integrity check

Before continuing processing, we check that this step has actually completed.

If `samtools quickcheck` completes without producing output, the header and end-of-file structure of both BAM files were readable.

In [ ]:
!samtools quickcheck -v "{ILL_BAM}" "{ONT_BAM}"
print("BAM quickcheck completed.")

## 59C. Exporting the outputs to a permanent location

We move the outputs from the temporary Colab space to a permanent location.

Colab's `/content` space is temporary. The cell below copies the quality-control reports, logs, reference files, and input manifest to Google Drive.

BAM and BAI files can be large. When `COPY_BAMS = True` is selected, they are exported too.

In [ ]:
COPY_BAMS = True
EXPORT_DIR = Path("/content/drive/MyDrive/BIF201_LMS_Alistirma_1_59"); EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for folder in (QC_DIR, LOG_DIR, REF_DIR): shutil.copytree(folder, EXPORT_DIR/folder.name, dirs_exist_ok=True)
if COPY_BAMS: [shutil.copy2(p, EXPORT_DIR/p.name) for p in (ILL_BAM, Path(str(ILL_BAM)+".bai"), ONT_BAM, Path(str(ONT_BAM)+".bai"))]
print("Permanent output folder:", EXPORT_DIR)

## 59D. Information useful when sharing a problem

When sharing a problem, having the following information in a single message helps us find the source of the error faster:

```text
Notebook version:
Exercise where the error occurred:
DATA_LOCATION value:
R1 path:
R2 path:
Nanopore path:
Full error output:
Last successful exercise before the error:
```

Along with a screenshot, saving the error text too makes it easier for us to jointly evaluate whether the issue lies in the file path, data structure, or command level.


# TODAY'S LIVE LESSON: EXERCISES 60-80

In exercises 1-59, we produced the reference, alignment, sorted BAM/BAI, and alignment quality-control outputs.

Today's live lesson continues from the end of these outputs: first we compare the coverage tables of the two platforms, then we move on to whole-genome variant calling on the natural data.

Before moving on to exercise 60, the startup check below verifies that the reference, the two BAM/BAI pairs, and the eight QC reports are present.


## Live-lesson startup check

Before producing new analysis, we verify the file contract we inherited from exercise 59. If this check fails, the missing file needs to be completed before moving on to variant calling.


In [ ]:
LIVE_INPUTS = [
    REFERENCE, Path(str(REFERENCE) + ".fai"),
    ILL_BAM, Path(str(ILL_BAM) + ".bai"),
    ONT_BAM, Path(str(ONT_BAM) + ".bai"),
]
LIVE_INPUTS += [QC_DIR / f"natural_{platform}.{report}"
                for platform in ("illumina", "nanopore")
                for report in ("flagstat.txt", "coverage.tsv", "idxstats.tsv", "stats.txt")]

live_audit = pd.DataFrame({
    "PATH": [str(p) for p in LIVE_INPUTS],
    "EXISTS": [p.exists() for p in LIVE_INPUTS],
    "SIZE_MB": [round(p.stat().st_size / 1e6, 2) if p.exists() else 0 for p in LIVE_INPUTS],
})
display(live_audit)
assert live_audit["EXISTS"].all(), "There is a missing input: check the table before moving on to exercise 60."
!samtools quickcheck -v "{ILL_BAM}" "{ONT_BAM}"
print("Live-lesson startup check completed.")


# PART VIII: Comparing the alignment results

Both platforms were aligned to the same reference. Read length, paired-end structure, supplementary records, and coverage distribution differ. The comparison is made while keeping in mind that the same metric doesn't mean the same thing on every technology.

## 60. Bringing the coverage tables into Python

Now we bring the SAMtools output into a Python table.

We will read the SAMtools coverage outputs as tab-separated tables.

In [ ]:
import pandas as pd
ill_cov = pd.read_csv(QC_DIR/"natural_illumina.coverage.tsv", sep="	")
ont_cov = pd.read_csv(QC_DIR/"natural_nanopore.coverage.tsv", sep="	")

## 61. Adding platform labels

To avoid mixing up the rows of the two platforms, we add source labels.

After merging, we'll keep track of which platform each row came from.

In [ ]:
ill_cov["platform"] = "Illumina"
ont_cov["platform"] = "Nanopore"

## 62. Building the coverage comparison table

Now we build a table that lets us read the results at a glance.

We'll show the four Illumina and four Nanopore rows in a single table.

In [ ]:
mapping_comparison = pd.concat([ill_cov, ont_cov], ignore_index=True)
display(mapping_comparison)

## 63. Saving the coverage comparison

We record this information now so we can trace it later.

We'll write the table in TSV format to the `comparison/` folder we created for this purpose.


In [ ]:
COMPARISON_TSV = COMP_DIR / "natural_mapping_comparison.tsv"
mapping_comparison.to_csv(COMPARISON_TSV, sep="\t", index=False)
print("Comparison table:", COMPARISON_TSV)


## Questions to consider together

At this point we're not just summing numbers. You can look for answers to the following questions using your own outputs:

1. How much of the reference was covered on each platform?
2. How was the average depth distributed across contigs?
3. To what extent was the Illumina paired-end structure preserved?
4. What biological or technical situations might the Nanopore supplementary records reflect?
5. Which quality indicators does each different read architecture highlight on the same reference?

# PART IX: Whole-genome variant calling on the natural data

In this part we will run both platforms across all four contigs. The Illumina call can be completed within the live lesson. The Nanopore pileup and call process can take a long time depending on Colab's resources.

Any Nanopore process not completed during the lesson can be continued at home or at the office. The same analytical scope is preserved without narrowing the genome region or subsampling the data.

We expect the number of high-confidence SNVs in the natural data to be very low. We'll read the produced candidates together with quality, depth, allele support, alignment context, and the platform's error profile.

## 64. Illumina natural-data call paths

First, in this step, we explicitly define the file paths we will use.

We'll keep the pileup, raw call, normalized SNV, and filtered SNV files separate.

In [ ]:
ILL_BCF, ILL_RAW = VAR_DIR/"natural_illumina.mpileup.bcf", VAR_DIR/"natural_illumina.raw.vcf.gz"
ILL_NORM, ILL_FILT = VAR_DIR/"natural_illumina.snvs.vcf.gz", VAR_DIR/"natural_illumina.filtered.snvs.vcf.gz"

## 65. Illumina whole-genome pileup

In this cell we collect the base evidence from the aligned reads along the reference coordinates.

Evidence below MAPQ 30 and base quality 25 is excluded. `-d 5000` caps the maximum read depth used within the pileup at a given position; if the real alignment depth is higher, DP/AD may be affected by this computational cap. The DP and AD fields will be used in call evaluation.


In [ ]:
!bcftools mpileup -Ob -o "{ILL_BCF}" -f "{REFERENCE}" -q 30 -Q 25 -d 5000 -a FORMAT/DP,FORMAT/AD "{ILL_BAM}"

## 66. Illumina haploid call

Now we call haploid variant candidates from the pileup evidence.

FLW-1 is modeled as a haploid bacterial sample. We'll only write the variant records to the VCF.

In [ ]:
!bcftools call --ploidy 1 -m -v -Oz -o "{ILL_RAW}" "{ILL_BCF}"

## 67. Illumina SNV normalization

In this step we bring the VCF records into a common representation relative to the reference.

We'll normalize the records against the reference and keep only biallelic SNV records.

In [ ]:
!bcftools norm -f "{REFERENCE}" -m -both -Ou "{ILL_RAW}" | bcftools view -v snps -m2 -M2 -Oz -o "{ILL_NORM}"

## 68. Illumina initial filter

Now we apply the initial filter that will form the first review set.

We'll build the first review set with QUAL 30 and total depth 10 thresholds. These thresholds do not replace final biological validation.

In [ ]:
!bcftools filter -i 'QUAL>=30 && INFO/DP>=10' -Oz -o "{ILL_FILT}" "{ILL_NORM}"

## 69. Indexing the Illumina filtered VCF

Now we build the corresponding index; the subsequent tools will build region access on this structure.

We'll build the Tabix index for regional queries and IGV.

In [ ]:
!bcftools index -f -t "{ILL_FILT}"

## 70. Illumina high-confidence SNV count

In this cell we compute the relevant record count.

A zero or very low count is consistent with the biological expectation. Results greater than zero are examined as candidates.

In [ ]:
print(ILL_FILT)
print("Does the file exist?", ILL_FILT.exists())

In [ ]:
!bcftools view -H "$ILL_FILT" | wc -l

In [ ]:
!bcftools view -H "$ILL_RAW" | wc -l

In [ ]:
!bcftools view -H "$ILL_NORM" | wc -l

70A. Directly reading the Illumina raw call record

Before moving on to the filtered SNV set, we'll directly inspect the records in the raw call file.

The `bcftools view -H` command removes the VCF header lines and shows only the variant records. This lets us see, without any field selection, how many candidates exist at the raw call stage and what information represents that candidate within the VCF.

In [ ]:
!bcftools view -H "$ILL_RAW"

## 71. Inspecting the Illumina raw candidate

No records remained in the filtered SNV set. So we'll open the single variant candidate found in the raw call file and evaluate which variant class it belongs to and why it was excluded during SNV selection.

The coordinate, reference and alternative allele, call quality, sample depth, and allele support will be read together.

In [ ]:
print("CHROM\tPOS\tREF\tALT\tQUAL\tSAMPLE_DP\tAD")
!bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%QUAL[\t%DP\t%AD]\n' "$ILL_RAW"

### How should we read the Illumina VCF candidate together?

- **CHROM + POS:** the candidate's reference coordinate
- **REF + ALT:** the base in the reference and the called alternative allele
- **QUAL:** the confidence score of the variant call; not the same metric as BaseQ and MAPQ
- **INFO/DP:** the total depth used during the call
- **AD:** the sample-level read depths supporting the reference and alternative allele

This list is used not as a results table, but as the first review layer where candidate evidence is read.


## 72. Runtime of the Nanopore whole-genome call

The Nanopore whole-genome pileup process can take a long time on a single processor core. This duration does not require us to narrow the analysis scope.

Run any process not completed during the live lesson at home or at the office before the next lesson. Keep the same four contigs and the same call parameters.

## 73. Nanopore whole-genome call paths

First, in this step, we explicitly define the file paths we will use.

We'll keep the pileup, raw call, normalized SNV, and filtered SNV files separate.

In [ ]:
ONT_BCF, ONT_RAW = VAR_DIR/"natural_nanopore.mpileup.bcf", VAR_DIR/"natural_nanopore.raw.vcf.gz"
ONT_NORM, ONT_FILT = VAR_DIR/"natural_nanopore.snvs.vcf.gz", VAR_DIR/"natural_nanopore.filtered.snvs.vcf.gz"

## 74. Nanopore whole-genome pileup

In this cell we collect the base evidence from the aligned reads along the reference coordinates.

We'll apply MAPQ 20 and base quality 15 thresholds. `-d 5000` caps the maximum read depth used within the pileup at a given position; since some regions in this dataset can exceed 5000x, the DP/AD values may be subject to this cap. We'll run the process across all four contigs.


In [ ]:
!bcftools mpileup -Ob -o "{ONT_BCF}" -f "{REFERENCE}" -q 20 -Q 15 -d 5000 -a FORMAT/DP,FORMAT/AD "{ONT_BAM}"

## 75. Nanopore whole-genome haploid call

Now we call haploid variant candidates from the pileup evidence.

We'll call the variant candidates across the four contigs with the haploid model.

In [ ]:
!bcftools call --ploidy 1 -m -v -Oz -o "{ONT_RAW}" "{ONT_BCF}"

## 76. Nanopore whole-genome SNV normalization

In this step we bring the VCF records into a common representation relative to the reference.

We'll normalize the records against the reference. We'll separate indels and build the biallelic SNV set.

In [ ]:
!bcftools norm -f "{REFERENCE}" -m -both -Ou "{ONT_RAW}" | bcftools view -v snps -m2 -M2 -Oz -o "{ONT_NORM}"

In [ ]:
!bcftools view -H "$ONT_NORM" | wc -l

## 77. Nanopore whole-genome initial filter

Now we apply the initial filter that will form the first review set.

We'll build the first review set with QUAL 30 and total depth 10 thresholds. These thresholds do not substitute for a platform-specific production filter.

In [ ]:
!bcftools filter -i 'QUAL>=30 && INFO/DP>=10' -Oz -o "{ONT_FILT}" "{ONT_NORM}"

## 78. Indexing the Nanopore whole-genome VCF

Now we build the corresponding index; the subsequent tools will build region access on this structure.

We'll index the filtered call set for regional queries and IGV.

In [ ]:
!bcftools index -f -t "{ONT_FILT}"

## 79. Nanopore whole-genome SNV count

In this cell we compute the relevant record count.

This number will give us the filtered SNV candidates across all four contigs. It's comparable to the Illumina call count over the same genome region.

In [ ]:
print("Nanopore filtered whole-genome SNVs:", end=" ")
!bcftools view -H "$ONT_FILT" | wc -l

## 80. Displaying the Nanopore whole-genome candidates

We bring the first candidates on screen together with quality, total depth, and allele support. Don't treat this list as a result; in the next step we'll read it together with context and the platform's error structure.

In [ ]:
print("CHROM\tPOS\tREF\tALT\tQUAL\tSAMPLE_DP\tAD")
!bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%QUAL[\t%DP\t%AD]\n' "$ONT_FILT" | head -n 20

## Exporting the live-lesson outputs to a permanent location

The Colab workspace is temporary. We'll copy the `comparison/` and `variants/` folders to the Google Drive space at the end of the lesson to preserve today's call outputs.


In [ ]:
LIVE_EXPORT_DIR = Path("/content/drive/MyDrive/BIF201_Canli_Ders_60_80")
LIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(COMP_DIR, LIVE_EXPORT_DIR / "comparison", dirs_exist_ok=True)
shutil.copytree(VAR_DIR, LIVE_EXPORT_DIR / "variants", dirs_exist_ok=True)
print("Live-lesson outputs:", LIVE_EXPORT_DIR)


## How should we read the natural-data results?

We produce both call sets on the same four contigs. So the call counts are based on the same genome region.

The general picture we expect:

- Zero or a low number of high-confidence SNVs in the Illumina call set
- A higher technical candidate load in the Nanopore call set

This difference may stem from measuring the same isolate with different read and error structures. A general-purpose caller responds differently to the two technical structures.

Four questions guide us for each candidate:

1. Is the depth sufficient?
2. How many reads support the alternative allele?
3. Is the position in a low-complexity or repetitive region?
4. Is the same signal also seen on the other platform?

# PART X: Controlled variant data assignment

The package I'll share contains the same FLW-1 reference and FASTQ files carrying 40 controlled SNVs.

In this work, the two platforms' data will be aligned separately, BAM and BAI files will be produced, alignment reports will be prepared, and whole-genome variant calls will be completed for both platforms.

The truth VCF is included in the package. Keeping the truth file closed until the calls are complete makes the evaluation more instructive. In the next live lesson we'll compare the VCF files together and measure the real performance.

# PART XI: Controlled whole-genome calls and evaluation

In this part, calls will be produced across the whole genome for both platforms. Long-running commands can be completed in your own working environment before the next lesson.

In the final live lesson we'll compare the ready VCF files with the truth VCF. We'll compute TP, FN, and genome-wide FP values, and evaluate sensitivity and precision over the same call region.


## How should we read the results together?

The whole-genome call table answers two questions at once:

1. How many of the known 40 SNVs were recovered?
2. How many additional candidates were produced outside the truth loci?

Sensitivity shows the recovery of the controlled SNVs; precision shows the whole-genome false-positive load. We'll compare the Illumina and Nanopore results over the same reference region.

# PART XII: IGV inspection

The numeric table gives us a summary of call performance. In IGV we'll see the read evidence at the same coordinate directly.

We select three record classes for inspection:

- TP: shared between the truth and call set
- FN: present in the truth but not called
- FP: present in the call set but outside the truth

# Closing

Throughout this work, by running two separate data sets on the same analytical backbone:

```text
Processed real reads
→ reference assembly
→ alignment
→ BAM quality control
→ natural whole-genome call signal

Reads carrying controlled SNVs
→ same reference assembly
→ separate platform alignments
→ whole-genome variant calling
→ truth comparison
→ TP, FN, and FP
→ sensitivity and precision
```

The first data set lets us see the limits of producing calls. The second data set makes call performance measurable.

Our real takeaway here, beyond memorizing individual commands, will be seeing how the reference, the FASTQ, the BAM, and the VCF connect to one another within the same decision chain.